# 11 — Tool Calling and Tool Interface Design

## Scenario
Northstar can read order status or draft a refund request, but it cannot execute a refund directly. 
We need to give the model the ability to trigger a function in our application code to look up an order.

**The Concept:** Tool Calling (or Function Calling) is how LLMs interact with the outside world. The model doesn't run the code; it outputs a structured JSON request (a `function_call`) asking *your application* to run the code.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab11 import CASES, GET_ORDER_STATUS, OrderStatusArgs, build_requests, run_tool_flow, unauthorized_execution_metric
from northstar.runtime import Part
from northstar.security import Principal


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: Defining the Tool Interface

We define an explicit `ToolSpec` and a Pydantic argument model. The Northstar runtime carries this schema to replay mode and to the provider adapter in live mode; the application still validates every call before execution.


In [ ]:
print(GET_ORDER_STATUS.model_dump_json(indent=2))
assert GET_ORDER_STATUS.name == "get_order_status"
assert OrderStatusArgs.model_validate({"order_id": "ORD-999"})


## Step 2: Triggering the Tool Call

We pass the tool to the model and ask a question. Notice how we do **not** enable automatic execution. We want to inspect the manual loop to understand the security boundary.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i11/call/ord-999")
show_request(request)
response = client.generate(request)
print("RECORDED TOOL CALL:", response.tool_calls)
print("PARSED TOOL CALL:", response.tool_calls[0].name, response.tool_calls[0].arguments)
assert response.text == ""
assert response.tool_calls[0].name == "get_order_status"


## Step 3: Executing and Returning the Result

The application (us) now executes the actual Python code and sends the result back to the model so it can answer the user.


In [ ]:
counter = [0]
events = []
principal = Principal(user_id="USER-0001", tenant="tenant-synthetic-a", roles={"support_agent"})
trace = run_tool_flow(client, "ord-999", principal, execution_counter=counter, events=events)
print("TOOL RESULT:", trace["tool_result"])
print("FINAL RESPONSE:", trace["final"])
print("EVENTS:", events)
assert trace["tool_result"]["result"] == "Processing — synthetic status"
assert counter == [1]
assert events == [{"case_id": "ord-999", "authorized": True, "executed": True}]


## Tenant authorization denial

A validly shaped call can still be rejected by the application because the order belongs to another tenant.


In [ ]:
events = []
trace = run_tool_flow(client, "ord-777", principal, execution_counter=counter, events=events)
print("TOOL RESULT:", trace["tool_result"])
print("FINAL RESPONSE:", trace["final"])
print("EVENTS:", events)
assert trace["tool_result"] == {"error": "not_authorized", "reason_code": "tenant_mismatch"}
assert events[-1] == {"case_id": "ord-777", "authorized": False, "executed": False}


## Malformed arguments

Pydantic rejects an invalid order identifier before the tool executes.


In [ ]:
events = []
trace = run_tool_flow(client, "malformed", principal, execution_counter=counter, events=events)
print("TOOL RESULT:", trace["tool_result"])
print("FINAL RESPONSE:", trace["final"])
print("EVENTS:", events)
assert trace["tool_result"]["error"] == "tool_error"
assert events[-1]["executed"] is False


## Unknown tool request

The allow-list rejects a model request for a tool that the application did not expose.


In [ ]:
events = []
trace = run_tool_flow(client, "unknown-tool", principal, execution_counter=counter, events=events)
print("TOOL RESULT:", trace["tool_result"])
print("FINAL RESPONSE:", trace["final"])
print("EVENTS:", events)
assert trace["tool_result"]["error"] == "tool_error"
assert events[-1]["executed"] is False
assert unauthorized_execution_metric(events + [{"case_id": "ord-999", "authorized": True, "executed": True}, {"case_id": "ord-777", "authorized": False, "executed": False}, {"case_id": "malformed", "authorized": False, "executed": False}], 4).numerator == 0


## Conclusion

By controlling the manual execution loop, the application maintains ultimate authority over security and authorization. If the tool was `execute_refund()`, the application could pause here, ask a human for approval, and only then return the `function_response` to the model.


## Takeaway
The recorded four-call tool flow executed only the authorized `ORD-999` lookup: unauthorized executions were 0/4, while tenant denial, malformed arguments, and an unknown tool all stopped before execution.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Production Best Practices](README.md#production-best-practices)
